# 00 — Data Understanding

**Goal:** get a precise picture of the raw dataset before any modeling — what's in it, what's
missing or invalid, and how we handle that (Assignment Tasks 1-3), plus a first pass at Task 4
(candidate features to hand off to feature engineering).

- **Input:** `data/raw/Uganda_Grid_Load_Dataset.csv`
- **Output:** `data/processed/grid_load_clean.csv`
- **Reusable code used here:** `src/data_loading.py`, `src/preprocessing.py`


In [1]:
# Standard imports. `src/` isn't an installed package, so we add the project root to
# sys.path to be able to `import src...` whether this notebook is run from `notebooks/`
# (normal case) or from the project root (e.g. via nbconvert --execute).
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loading import load_dataset
from src.preprocessing import audit_missing_values, clean_data

pd.set_option("display.max_columns", None)


## Load the raw dataset

In [2]:
# load_dataset() reads the CSV with no cleaning applied, so every dirty value below is
# exactly what's in the file on disk.
raw_path = PROJECT_ROOT / "data" / "raw" / "Uganda_Grid_Load_Dataset.csv"
df_raw = load_dataset(raw_path)

print(f"shape: {df_raw.shape}")
df_raw.head()


shape: (1490, 11)


,Record_ID,Region,Hour,DayOfWeek,Temperature_C,Humidity_pct,Rainfall_mm,PopulationIndex,IndustrialIndex,SolarGenerationIndex,GridLoad_MW
0,1,Central,20,0,hot,52.0,3.1,93,89,0.03,850.3
1,2,Eastern,13,0,18.5,NaN,3.7,95,89,0.38,788.3
2,3,Western,14,4,22.7,90.0,unknown,71,55,0.13,548.0
3,4,Northern,10,0,19.6,41.0,0.1,###,78,0.45,604.6
4,5,Northern,12,0,27.4,88.0,0.0,86,high,0.70,546.1


In [3]:
# Every column below that should be numeric but shows up as `object` dtype has at least one
# non-numeric value mixed in — that's the first clue of what the audit needs to find.
df_raw.dtypes


Record_ID                 int64
Region                      str
Hour                      int64
DayOfWeek                 int64
Temperature_C               str
Humidity_pct            float64
Rainfall_mm                 str
PopulationIndex             str
IndustrialIndex             str
SolarGenerationIndex    float64
GridLoad_MW                 str
dtype: object

## Task 1 — Missing / invalid value audit

`df.isna().sum()` alone would undersell the problem here: several columns hide bad values
behind placeholder strings (`"N/A"`, `"unknown"`, `"###"`, ...) or a stray unit-prefixed number
(`"MW900"`) instead of a real `NaN`. A few columns also carry values that parse fine as numbers
but are physically impossible (e.g. an `Hour` of 30, negative humidity). Both cases mean
"we don't have a usable reading" for that cell, so the audit below treats both as missing.


In [4]:
# Column-by-column: what does a numeric column look like where a direct numeric parse fails?
# This is exactly what justifies the token list in src/preprocessing.py (MISSING_TOKENS)
# instead of guessing at it.
numeric_like = [
    "Hour", "DayOfWeek", "Temperature_C", "Humidity_pct", "Rainfall_mm",
    "PopulationIndex", "IndustrialIndex", "SolarGenerationIndex", "GridLoad_MW",
]

for col in numeric_like:
    bad_values = df_raw.loc[pd.to_numeric(df_raw[col], errors="coerce").isna(), col].unique()
    if len(bad_values):
        print(f"{col}: {sorted(bad_values.tolist(), key=str)}")


Temperature_C: ['hot', nan]
Humidity_pct: [nan]
Rainfall_mm: [nan, 'unknown']
PopulationIndex: ['###']
IndustrialIndex: ['high']
SolarGenerationIndex: [nan]
GridLoad_MW: ['MW900']


In [5]:
# Exact duplicate rows (same Record_ID and every other value repeated) inflate the dataset
# without adding information, and would leak between train/test splits later if left in.
n_duplicates = df_raw.duplicated().sum()
print(f"exact duplicate rows: {n_duplicates}")
df_raw[df_raw.duplicated(keep=False)].sort_values("Record_ID").head(4)


exact duplicate rows: 10


,Record_ID,Region,Hour,DayOfWeek,Temperature_C,Humidity_pct,Rainfall_mm,PopulationIndex,IndustrialIndex,SolarGenerationIndex,GridLoad_MW
37,38,Eastern,21,0,24.0,70.0,6.8,65,40,0.24,525.8
1480,38,Eastern,21,0,24.0,70.0,6.8,65,40,0.24,525.8
53,54,Northern,23,6,26.1,51.0,3.1,99,86,0.04,767.2
1484,54,Northern,23,6,26.1,51.0,3.1,99,86,0.04,767.2


In [6]:
# Task 1 deliverable: missing/invalid count and percentage per feature. audit_missing_values()
# applies the same placeholder-token + valid-range rules that clean_data() will use to fix
# things, so this table is a preview of exactly what cleaning is about to touch.
audit = audit_missing_values(df_raw)
audit


,column,missing_count,missing_pct
0,Rainfall_mm,4,0.27
1,Temperature_C,2,0.13
2,Humidity_pct,2,0.13
3,SolarGenerationIndex,2,0.13
4,Hour,1,0.07
5,PopulationIndex,1,0.07
6,IndustrialIndex,1,0.07
7,Record_ID,0,0.00
8,Region,0,0.00
9,DayOfWeek,0,0.00


**Findings:**

- Every numeric column carries exactly one seeded placeholder token (`"hot"`, `"N/A"`,
  `"unknown"`, `"###"`, `"high"`) or a unit-prefixed number (`"MW900"`), plus a handful of
  blank cells in `Temperature_C`, `Rainfall_mm`, and `SolarGenerationIndex`.
- `Hour` has one out-of-range value (30, outside the valid 0-23), and `Humidity_pct` /
  `Rainfall_mm` each have physically impossible negative readings.
- 10 rows are exact duplicates of another row.
- No single column exceeds roughly 1% missing/invalid data.

## Task 2 — Deciding how to handle missing data

- **Imputation over removal:** with well under 1% missing per column, dropping affected rows
  would throw away otherwise-good data (e.g. a row missing only `Temperature_C` still has a
  perfectly good `GridLoad_MW`) for very little benefit.
- **Median as the default strategy:** robust to the skew/outliers already visible in columns
  like `Rainfall_mm`, and simple to justify and reproduce. `src/preprocessing.clean_data()`
  also supports `impute_strategy="knn"` as an alternative worth comparing against median in an
  `imputation_*` experiment notebook — median is the safe default, KNN is the thing to test.
- **Duplicates are dropped outright**, not imputed — they carry no new information.


## Task 3 — Implement cleaning and evaluate its impact

In [7]:
# clean_data() drops duplicates, coerces + range-checks each numeric column, and imputes
# what's left using the median. It returns a report alongside the cleaned data so the effect
# of each step is visible rather than silent.
df_clean, cleaning_report = clean_data(df_raw, impute_strategy="median")

print(f"duplicates dropped: {cleaning_report.attrs['duplicates_dropped']}")
print(f"rows before: {len(df_raw)} -> rows after: {len(df_clean)}")
cleaning_report


duplicates dropped: 10
rows before: 1490 -> rows after: 1480


,column,missing_before,missing_after
0,Hour,1,0
1,DayOfWeek,0,0
2,Temperature_C,2,0
3,Humidity_pct,2,0
4,Rainfall_mm,4,0
5,PopulationIndex,1,0
6,IndustrialIndex,1,0
7,SolarGenerationIndex,2,0
8,GridLoad_MW,0,0


Note `GridLoad_MW` shows `missing_before = 0` even though the audit above flagged
`"MW900"` as a bad token — that value was recovered directly (`"MW900"` -> `900`) by the
regex fallback in `_coerce_numeric` during coercion itself, before the missing-value count is
taken. It never became a gap that needed imputing, unlike `"hot"` or `"###"`, which have no
number to recover and do go through imputation.


In [8]:
# Confirm the cleaning step actually did its job: no missing values left in any column that
# had them, and every previously-object numeric column is now a real numeric dtype.
touched_cols = cleaning_report["column"].tolist()
assert df_clean[touched_cols].isna().sum().sum() == 0, "cleaning left NaNs behind"
df_clean.dtypes


Record_ID                 int64
Region                      str
Hour                    float64
DayOfWeek                 int64
Temperature_C           float64
Humidity_pct            float64
Rainfall_mm             float64
PopulationIndex         float64
IndustrialIndex         float64
SolarGenerationIndex    float64
GridLoad_MW             float64
dtype: object

In [9]:
# Evaluate impact (Task 3): compare summary stats before vs after cleaning for the columns
# that actually had missing/invalid values, to check median imputation didn't visibly distort
# each distribution (a handful of imputed values out of ~1490 rows should barely move mean/std).
dirty_cols = cleaning_report.loc[cleaning_report["missing_before"] > 0, "column"].tolist()

before_stats = df_raw[dirty_cols].apply(pd.to_numeric, errors="coerce").describe().T
after_stats = df_clean[dirty_cols].describe().T
comparison = before_stats.join(after_stats, lsuffix="_before", rsuffix="_after")
comparison[["mean_before", "mean_after", "std_before", "std_after"]]


,mean_before,mean_after,std_before,std_after
Hour,11.990604,11.928378,7.079415,7.055580
Temperature_C,26.515390,26.528446,4.971249,4.975564
Humidity_pct,64.055742,64.066892,17.659648,17.544349
Rainfall_mm,4.097445,4.088919,3.985879,3.955751
PopulationIndex,75.134318,75.145946,14.548887,14.549779
IndustrialIndex,57.871054,57.867568,21.754854,21.754983
SolarGenerationIndex,0.245901,0.246128,0.220371,0.220603


## Save the cleaned dataset

In [10]:
processed_path = PROJECT_ROOT / "data" / "processed" / "grid_load_clean.csv"
df_clean.to_csv(processed_path, index=False)
print(f"saved: {processed_path.relative_to(PROJECT_ROOT)}  shape={df_clean.shape}")


saved: data/processed/grid_load_clean.csv  shape=(1480, 11)


## Task 4 — Candidate features (preview only)

Ideas to hand off to whoever owns `feature_engineering_*` experiment notebooks — not
implemented here, since Task 4 only asks to *identify* candidates:

- **Cyclical encoding of `Hour`** (sin/cos) so hour 23 and hour 0 are numerically adjacent.
- **`is_weekend` flag** from `DayOfWeek`, since weekday/weekend demand often differs.
- **One-hot encoding of `Region`** for the tree-based/linear models in `docs/workflow.md` §7.
- **Temperature × Humidity interaction** — cooling/heating demand often depends on both jointly,
  not either alone.
- **Hour × IndustrialIndex interaction** — industrial draw is plausibly time-of-day dependent.

## Next steps

- Log this cleaning pass in `experiment_log.md`.
- Continue to `01_eda.ipynb` for visual exploration of `data/processed/grid_load_clean.csv`.
